## SW4

In [2]:
from pathlib import Path
import subprocess
import textwrap


class SW4Run:
    def __init__(self,
                 path: str,
                 nodes: int = 7,
                 tasks_per_node: int = 16,
                 job_name_level: int = 0,
                 exclude_nodes: list[str] | None = None,
                 sw4_exe: str = "sw4",
                 lib_path: str = "/mnt/nfshare/lib_local",
                 verbose: bool = True):

        self.path = Path(path).resolve()
        self.nodes = nodes
        self.tasks_per_node = tasks_per_node
        self.job_name_level = job_name_level
        self.exclude = exclude_nodes
        self.sw4_exe = sw4_exe
        self.lib_path = lib_path
        self.verbose = verbose

        self.runs = sorted(self.path.rglob("*.in"))

        if not self.runs:
            raise FileNotFoundError(f"No .in files found in: {self.path}")

    def _job_name(self, in_file: Path) -> str:
        folder = in_file.parent
        level = abs(self.job_name_level)
        for _ in range(level):
            folder = folder.parent
        return folder.name

    def _build_script(self, in_file: Path) -> Path:
        job_name = self._job_name(in_file)

        header = [
            "#!/bin/bash",
            f"#SBATCH --job-name={job_name}",
            f"#SBATCH --nodes={self.nodes}",
            f"#SBATCH --tasks-per-node={self.tasks_per_node}",
            f"#SBATCH --output=log_{job_name}.log",
        ]

        if self.exclude:
            header.append(f"#SBATCH --exclude={','.join(self.exclude)}")

        body = textwrap.dedent(f"""\

            pwd; hostname; date
            SECONDS=0

            export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:{self.lib_path}
            mpirun {self.sw4_exe} {in_file.name}

            echo "Elapsed: $SECONDS seconds."
            date
        """)

        script_path = in_file.parent / "run.sh"
        script_path.write_text("\n".join(header) + "\n" + body)
        script_path.chmod(0o755)

        if self.verbose:
            print(f"📝 {script_path} (job={job_name})")

        return script_path

    def submit(self):
        job_ids = []
        for in_file in self.runs:
            script = self._build_script(in_file)
            proc = subprocess.run(
                ["sbatch", str(script)],
                capture_output=True, text=True, check=True,
                cwd=in_file.parent
            )
            job_id = int(proc.stdout.split()[-1])
            job_ids.append(job_id)
            if self.verbose:
                print(f"🚀 Job {job_id} submitted from {in_file.parent}")
        return job_ids

In [4]:
SW4Run(
    # path="/mnt/deadmanschest/pxpalacios/SW4/shakermaker_2_sw4/test_LOH/New folder/test_DD",
    path = '/mnt/leviathanchest/pxpalacios/sw4_STG_test',
    nodes=15,
    tasks_per_node=16,
    job_name_level=-1,
    exclude_nodes= None,
    sw4_exe = "sw4",
    lib_path = "/mnt/nfshare/lib_local",
    verbose = True,
).submit()

📝 /mnt/leviathanchest/pxpalacios/sw4_STG_test/LOH_source_STG_2sources/sw4/run.sh (job=LOH_source_STG_2sources)
🚀 Job 143021 submitted from /mnt/leviathanchest/pxpalacios/sw4_STG_test/LOH_source_STG_2sources/sw4
📝 /mnt/leviathanchest/pxpalacios/sw4_STG_test/LOH_source_STG_TOPO_2sources/sw4/run.sh (job=LOH_source_STG_TOPO_2sources)
🚀 Job 143022 submitted from /mnt/leviathanchest/pxpalacios/sw4_STG_test/LOH_source_STG_TOPO_2sources/sw4
📝 /mnt/leviathanchest/pxpalacios/sw4_STG_test/ffsp_source_STG_2sources/sw4/run.sh (job=ffsp_source_STG_2sources)
🚀 Job 143023 submitted from /mnt/leviathanchest/pxpalacios/sw4_STG_test/ffsp_source_STG_2sources/sw4
📝 /mnt/leviathanchest/pxpalacios/sw4_STG_test/ffsp_source_STG_TOPO_2sources/sw4/run.sh (job=ffsp_source_STG_TOPO_2sources)
🚀 Job 143024 submitted from /mnt/leviathanchest/pxpalacios/sw4_STG_test/ffsp_source_STG_TOPO_2sources/sw4


[143021, 143022, 143023, 143024]

In [ ]:
ppp

In [ ]:
import tarfile
import os
import sys

def extract_tar_gz(archive_path, output_dir=None):
    if not os.path.isfile(archive_path):
        print(f"❌ File not found: {archive_path}")
        return

    if not archive_path.endswith(".tar.gz"):
        print(f"❌ Not a .tar.gz file: {archive_path}")
        return

    if output_dir is None:
        output_dir = os.path.dirname(archive_path)

    try:
        print(f"📦 Extracting: {archive_path}")
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(path=output_dir)
        print(f"✅ Successfully extracted to: {output_dir}")
    except Exception as e:
        print(f"⚠️ Extraction failed: {e}")

# === Run from command line ===
if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python extract_tar.py /path/to/archive.tar.gz [output_folder]")
        sys.exit(1)

    archive = sys.argv[1]
    output = sys.argv[2] if len(sys.argv) > 2 else None
    extract_tar_gz(archive, output)


In [ ]:
extract_tar_gz('/mnt/leviathanchest/pxpalacios/sw4/STG_LOH1_topo.tar.gz')

In [ ]:
extract_tar_gz('/mnt/leviathanchest/pxpalacios/sw4/ffsp_source.tar.gz')